In [1]:
import os
import pandas as pd
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geo_root = repo_root.parent.parent / 'Data' / 'Geospatial'

# API key: c2ccd182-be71-41ee-6a99-08de61444905
# Another: 1de25faa-0796-4036-6a9c-08de61444905
# Another: 80b20037-8e82-42ec-6a9f-08de61444905
# Another: a2dbb3a9-026c-4aa5-6aa0-08de61444905
# Another: ed928d84-8050-4861-6aa1-08de61444905

import requests
from pandas import json_normalize
import time

#API_KEYS = ['1de25faa-0796-4036-6a9c-08de61444905', '80b20037-8e82-42ec-6a9f-08de61444905', 'a2dbb3a9-026c-4aa5-6aa0-08de61444905', 'ed928d84-8050-4861-6aa1-08de61444905', 'c2ccd182-be71-41ee-6a99-08de61444905']
#API_KEYS = ["01fa8599-3e8c-48ef-6aa2-08de61444905",'a2dbb3a9-026c-4aa5-6aa0-08de61444905', 'ed928d84-8050-4861-6aa1-08de61444905']
API_KEYS = ['49ae6bb0-939b-47f7-2514-08de74442a35', 'c2ccd182-be71-41ee-6a99-08de61444905']

API_BASE = "https://bdl.stat.gov.pl/api/v1"
API_KEY = API_KEYS[0]
# os.getenv("BDL_API_KEY")  # optional: export BDL_API_KEY to raise rate limits

HEADERS = {"User-Agent": "LRDWI-Notebook/1.0"}
if API_KEY:
    HEADERS["X-ClientId"] = API_KEY

OUTPUT_DIR = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS" / "metadata"
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS" / "data"

# Install nest_asyncio if not available
try:
    import nest_asyncio
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio", "aiohttp"])
    import nest_asyncio

In [2]:
# Censuses variables codes:
# NSP1988: age: "P2884", educ: "P2885, sex: "P2883", hh_size: "P2887"
# NSP2002: age x sex: "P2114", age x educ: "P2403", sex x educ: "P2402", hh_size: "P2871"
# NSP2011: age x sex: "P3304", age x educ: "P3311", sex x educ: "P3309", educ x city-rural: "P3310", hh_size: "P3420"
# NSP2021: age x sex: "P4253", age x educ: "P4320", sex x educ x city-rural: "P4345", hh_size: "P4287"

census_vars = {
    "NSP1988": {
        "age": "P2884",
        "educ": "P2885",
        "sex x city-rural": "P2883",
        "hh_size": "P2887"
    },
    "NSP2002": {
        "age x sex": "P2114",
        "age x educ": "P2403",
        "sex x educ": "P2402",
        "hh_size": "P2871"
    },
    "NSP2011": {
        "age x sex": "P3304",
        "age x educ": "P3311",
        "sex x educ": "P3309",
        "educ x city-rural": "P3310",
        "hh_size": "P3420"
    },
    "NSP2021": {
        "age x sex": "P4253",
        "age x educ": "P4320",
        "sex x educ x city-rural": "P4345",
        "hh_size": "P4287"
    }
}

In [ ]:
ENDPOINT = f"{API_BASE}/variables/search"

session = requests.Session()
session.headers.update(HEADERS)

census_variables = {}
page = 0
page_size = 100   # large page to minimize requests; API supports paging

for keys, dict in census_vars.items():
    print(f"Downloading census variables for {keys}...")
    df_census = pd.DataFrame()
    for name, var_code in dict.items():
        rows = []
        print(f" Variable '{name}' (code: {var_code}):")
        page = 0
        while True:
            params = {
                "subject-id": var_code,
                "page": page,
                "page-size": page_size,
                "format": "json",
                "sort": "Id",
            }
            r = session.get(ENDPOINT, params=params, timeout=30)
            if r.status_code == 412 and not API_KEY:
                if page_size > 200:
                    page_size = 200
                    print("Received 412 (Precondition Failed). Retrying with page_size=200. Set BDL_API_KEY to avoid this.")
                    time.sleep(0.2)
                    continue
                raise SystemExit("BDL API returned 412 (Precondition Failed). Set env var BDL_API_KEY with your X-ClientId.")
            r.raise_for_status()
            js = r.json()
            if page == 0:
                print(f"Total records: {js.get("totalRecords")}")
            
            results = js.get("results",[])
            
            if not results:
                break
            rows.extend(results)
            
            print(f"  page {page} -> got {len(results)} variables; total so far: {len(rows)}")
            
            page += 1
            time.sleep(0.05)
        # flatten JSON objects into a DataFrame (safe: different variables may have different nested fields)
        if not rows:
            raise SystemExit("No variables returned. Check network / API / credentials.")
        df = json_normalize(rows, sep="_")
        df_census = pd.concat([df_census, df], ignore_index=True)

    census_variables[keys] = df_census
    

In [ ]:
# Save census variables metadata to outputdir
for census_name, df_var in census_variables.items():
    output_path = OUTPUT_DIR / f"census_{census_name}_variables_metadata.csv"
    df_var.to_csv(output_path, index=False)
    print(f"Saved metadata for {census_name} to {output_path}")

In [3]:
# Load census variables metadata from outputdir
census_variables = {}
for census_name in census_vars.keys():
    input_path = OUTPUT_DIR / f"census_{census_name}_variables_metadata.csv"
    df_var = pd.read_csv(input_path)
    census_variables[census_name] = df_var

In [4]:
# Test the api keys by requesting https://bdl.stat.gov.pl/api/v1/version and reading the RESPONSE HEADERS.
# We print them for every API key.
ENDPOINT = f"{API_BASE}/version"

for api_key in API_KEYS:
    headers = HEADERS.copy()
    headers["X-ClientId"] = api_key
    r = requests.get(ENDPOINT, headers=headers)
    print(f"API Key: {api_key}")
    r.raise_for_status()
    print(r.headers)
    

API Key: 49ae6bb0-939b-47f7-2514-08de74442a35
{'Cache-Control': 'public,max-age=1800', 'Content-Length': '75', 'Content-Type': 'application/json; charset=utf-8', 'ETag': 'Qkmb1o4XYkCXQbWDIcLeGw', 'Vary': 'Accept,Accept-Language', 'X-Rate-Limit-Limit': '7d', 'X-Rate-Limit-Remaining': '49999', 'X-Rate-Limit-Reset': '2026-03-04T21:21:37.7824980Z', 'Date': 'Wed, 25 Feb 2026 21:21:37 GMT', 'Set-Cookie': 'cookie=85e497491d001d5aeeddc2fbe2679f69;max-age=1800;Path=/;HttpOnly, cookiesession1=678B2889B13FEF2951F32AFDE30B31F9;Expires=Thu, 25 Feb 2027 21:21:38 GMT;Path=/;HttpOnly'}
API Key: c2ccd182-be71-41ee-6a99-08de61444905
{'Cache-Control': 'public,max-age=1800', 'Content-Length': '75', 'Content-Type': 'application/json; charset=utf-8', 'Age': '1', 'ETag': 'Qkmb1o4XYkCXQbWDIcLeGw', 'Vary': 'Accept,Accept-Language', 'X-Rate-Limit-Limit': '7d', 'X-Rate-Limit-Remaining': '49999', 'X-Rate-Limit-Reset': '2026-03-04T21:21:39.0396821Z', 'Date': 'Wed, 25 Feb 2026 21:21:38 GMT', 'Set-Cookie': 'cookie=8

In [5]:
import asyncio
import aiohttp
from typing import List, Dict, Any
import warnings

# Suppress asyncio warnings in Jupyter
warnings.filterwarnings('ignore', category=RuntimeWarning, message='coroutine.*was never awaited')

async def fetch_page(session: aiohttp.ClientSession, endpoint: str, page: int, 
                     page_size: int, params_base: Dict, variable_id: int, 
                     api_key_index: int = 0) -> tuple[int, List[Dict[Any, Any]], bool]:
    """Fetch a single page asynchronously with API key rotation on failure"""
    params = {
        **params_base,
        "page": page,
        "page-size": page_size,
        "format": "json"
    }
    
    max_retries = len(API_KEYS)
    current_key_index = api_key_index
    
    for attempt in range(max_retries):
        try:
            # Create new headers for this request
            headers = {"User-Agent": "LRDWI-Notebook/1.0", "X-ClientId": API_KEYS[current_key_index]}
            
            async with session.get(endpoint, params=params, headers=headers) as response:
                if response.status == 412:
                    print(f"Variable {variable_id}, page {page}: Received 412, trying next API key")
                    current_key_index = (current_key_index + 1) % len(API_KEYS)
                    continue
                    
                response.raise_for_status()
                js = await response.json()
                return page, js.get("results", []), True
                
        except Exception as e:
            print(f"Variable {variable_id}, page {page}: Error with API key {current_key_index}: {str(e)}")
            current_key_index = (current_key_index + 1) % len(API_KEYS)
            
            if attempt == max_retries - 1:
                print(f"FAILED after trying all API keys - var_id: {variable_id}, page: {page}, page_size: {page_size}")
                return page, [], False
            
            await asyncio.sleep(0.1)  # Small delay between retries
    
    return page, [], False

async def download_variable_data_async(variable_id: int, subject_id: str, level: int) -> pd.DataFrame:
    """Download all data for a given variable_id using async requests (max 10 concurrent)"""
    ENDPOINT = f"{API_BASE}/data/by-variable/{variable_id}"
    
    params_base = {}
    page_size = 100
    all_data = []
    current_api_key_index = 1
    
    # Use connector with proper settings to avoid warnings
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=10)
    timeout = aiohttp.ClientTimeout(total=300)
    headers = {"User-Agent": "LRDWI-Notebook/1.0", "X-ClientId": API_KEYS[current_api_key_index]}
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=headers) as session:
        try:
            # Get first page to determine total
            params = {**params_base, "page": 0, "page-size": page_size, "format": "json"}
            async with session.get(ENDPOINT, params=params) as response:
                if response.status == 412:
                    page_size = 200
                    print(f"Variable {variable_id}: Received 412. Adjusting page_size to 200.")
                    current_api_key_index = (current_api_key_index + 1) % len(API_KEYS)
                    headers["X-ClientId"] = API_KEYS[current_api_key_index]
                    params["page-size"] = page_size
                    
                    async with session.get(ENDPOINT, params=params, headers=headers) as retry_response:
                        retry_response.raise_for_status()
                        js = await retry_response.json()
                else:
                    response.raise_for_status()
                    js = await response.json()
                    
                total_records = js.get("totalRecords", 0)
                print(f"  Total records for variable {variable_id}: {total_records}; level: {level}")
                all_data.extend(js.get("results", []))
                
        except Exception as e:
            print(f"  Variable {variable_id}: Failed to get initial data: {str(e)}")
            print(f"  FAILED - var_id: {variable_id}, page: 0, page_size: {page_size}")
            return pd.DataFrame()
        
        if not all_data:
            print(f"  No data for variable {variable_id}")
            return pd.DataFrame()
        
        page = 1
        total_pages = (total_records + page_size - 1) // page_size
        
        # Download remaining pages in batches of 10
        while page < total_pages:
            batch_pages = list(range(page, min(page + 10, total_pages)))
            
            # Fetch batch concurrently
            tasks = [fetch_page(session, ENDPOINT, p, page_size, params_base, variable_id, current_api_key_index) 
                    for p in batch_pages]
            results = await asyncio.gather(*tasks, return_exceptions=True)
            
            # Process results
            empty_count = 0
            failed_count = 0
            for result in results:
                if isinstance(result, Exception):
                    failed_count += 1
                    continue
                    
                page_num, page_results, success = result
                if not success:
                    failed_count += 1
                    print(f"  Skipping page {page_num} due to persistent failures")
                elif not page_results:
                    empty_count += 1
                else:
                    all_data.extend(page_results)
                    print(f"  page {page_num} -> got {len(page_results)} records; total so far: {len(all_data)}")
            
            # If all requests in the batch were empty, we're done
            if empty_count == len(batch_pages):
                print(f"  All {empty_count} pages in batch were empty. Stopping.")
                break
            
            # If too many failures, consider stopping
            if failed_count > 5:
                print(f"  Too many failures ({failed_count}) in this batch. Stopping.")
                break
            
            page += 10
    
    if not all_data:
        return pd.DataFrame()
        
    
    # Verify we got all expected records
    if len(all_data) != total_records:
        print(f"  WARNING: Expected {total_records} records but got {len(all_data)}")
        
    df_data = json_normalize(all_data, sep="_")
    df_data['variableId'] = variable_id
    df_data['subjectId'] = subject_id
    print(f"  ✓ Successfully collected {len(all_data)} records (expected: {total_records})")
    return df_data

def download_variable_data(variable_id: int, subject_id: str, level: int) -> pd.DataFrame:
    """Synchronous wrapper for async download function"""
    try:
        # Check if event loop is already running
        loop = asyncio.get_running_loop()
        # We're in Jupyter - use nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()
        # Create new event loop for this task to avoid context issues
        new_loop = asyncio.new_event_loop()
        asyncio.set_event_loop(new_loop)
        try:
            return new_loop.run_until_complete(download_variable_data_async(variable_id, subject_id, level))
        finally:
            new_loop.close()
    except RuntimeError:
        # No running loop, safe to create one
        return asyncio.run(download_variable_data_async(variable_id, subject_id, level))

In [6]:
# Download census data for each wave separately with incremental saving
# Each census wave will be saved to a separate parquet file

# Create output directory if it doesn't exist
census_data_dir = gus_root / "census_data"
census_data_dir.mkdir(parents=True, exist_ok=True)

# Download data for each census wave
for census_wave, df_census_vars in census_variables.items():
    print(f"\n{'='*80}")
    print(f"Processing {census_wave} - {len(df_census_vars)} variables to download")
    print(f"{'='*80}\n")
    
    # Initialize DataFrame for this census wave
    df_census_data = pd.DataFrame()
    output_file = census_data_dir / f"{census_wave}_data.parquet"
    
    # Check if file already exists and load existing data
    if output_file.exists():
        try:
            df_census_data = pd.read_parquet(output_file)
            existing_var_ids = set(df_census_data['variableId'].unique()) if 'variableId' in df_census_data.columns else set()
            print(f"Found existing file with {len(existing_var_ids)} variables already downloaded")
            print(f"Variables already in file: {sorted(existing_var_ids)}\n")
        except Exception as e:
            print(f"Warning: Could not load existing file: {e}")
            print("Starting fresh download\n")
            df_census_data = pd.DataFrame()
            existing_var_ids = set()
    else:
        existing_var_ids = set()
    
    # Download each variable
    success_count = 0
    failed_vars = []
    
    for idx, row in df_census_vars.iterrows():
        var_id = row['id']
        subject_id = row['subjectId']
        var_name = row.get('n5', 'Unknown')
        level = row.get('level', 6)  # Default to level 6 if not specified
        level = int(level) if pd.notna(level) else 6
        level = int(max(1, min(level, 6)))  # Ensure level is between 1 and 6
        
        # Skip if already downloaded
        if var_id in existing_var_ids:
            print(f"[{idx+1}/{len(df_census_vars)}] Variable {var_id} ({subject_id}): Already downloaded, skipping")
            success_count += 1
            continue
        
        print(f"[{idx+1}/{len(df_census_vars)}] Downloading variable {var_id} ({subject_id}): {var_name}")
        
        try:
            # Download variable data
            df_var_data = download_variable_data(var_id, subject_id, level)
            
            if df_var_data.empty:
                print(f"  WARNING: No data returned for variable {var_id}")
                failed_vars.append((var_id, subject_id, "No data returned"))
            else:
                # Append to census data
                df_census_data = pd.concat([df_census_data, df_var_data], ignore_index=True)
                success_count += 1
                
                # Save incrementally after each variable
                try:
                    df_census_data.to_parquet(output_file, index=False, compression='snappy')
                    print(f"  ✓ Variable {var_id} downloaded ({len(df_var_data)} records). Total records: {len(df_census_data)}")
                except Exception as save_error:
                    print(f"  WARNING: Data downloaded but failed to save: {save_error}")
                    # Also try CSV as backup
                    try:
                        backup_file = census_data_dir / f"{census_wave}_data_backup.csv"
                        df_census_data.to_csv(backup_file, index=False, encoding='utf-8')
                        print(f"  Saved backup to CSV: {backup_file}")
                    except:
                        pass
        
        except Exception as e:
            print(f"  ERROR: Failed to download variable {var_id}: {str(e)}")
            failed_vars.append((var_id, subject_id, str(e)))
        
        # Small delay between variables to avoid overwhelming the API
        time.sleep(0.1)
    
    # Final save for this census wave
    print(f"\n{'-'*80}")
    print(f"Completed {census_wave}:")
    print(f"  Successfully downloaded: {success_count}/{len(df_census_vars)} variables")
    print(f"  Total records: {len(df_census_data)}")
    print(f"  Saved to: {output_file}")
    
    if failed_vars:
        print(f"  Failed variables ({len(failed_vars)}):")
        for var_id, subject_id, error in failed_vars:
            print(f"    - Variable {var_id} ({subject_id}): {error}")
    
    # Also save as CSV for compatibility
    csv_file = census_data_dir / f"{census_wave}_data.csv"
    try:
        df_census_data.to_csv(csv_file, index=False, encoding='utf-8')
        print(f"  Also saved CSV to: {csv_file}")
    except Exception as e:
        print(f"  Warning: Could not save CSV: {e}")
    
    print(f"{'-'*80}\n")

print("\n" + "="*80)
print("ALL CENSUS WAVES COMPLETED")
print("="*80)



Processing NSP1988 - 19 variables to download

[1/19] Downloading variable 196133 (P2884): Unknown
  Total records for variable 196133: 4116; level: 6
  page 1 -> got 100 records; total so far: 200
  page 2 -> got 100 records; total so far: 300
  page 3 -> got 100 records; total so far: 400
  page 4 -> got 100 records; total so far: 500
  page 5 -> got 100 records; total so far: 600
  page 6 -> got 100 records; total so far: 700
  page 7 -> got 100 records; total so far: 800
  page 8 -> got 100 records; total so far: 900
  page 9 -> got 100 records; total so far: 1000
  page 10 -> got 100 records; total so far: 1100
  page 11 -> got 100 records; total so far: 1200
  page 12 -> got 100 records; total so far: 1300
  page 13 -> got 100 records; total so far: 1400
  page 14 -> got 100 records; total so far: 1500
  page 15 -> got 100 records; total so far: 1600
  page 16 -> got 100 records; total so far: 1700
  page 17 -> got 100 records; total so far: 1800
  page 18 -> got 100 records; to

In [ ]:
### PROBLEM from last:
'''
[195/247] Downloading variable 396075 (P3309): Unknown
  Variable 396075: Failed to get initial data: Cannot connect to host bdl.stat.gov.pl:443 ssl:default [Connect call failed ('194.165.48.118', 443)]
  FAILED - var_id: 396075, page: 0, page_size: 100
  WARNING: No data returned for variable 396075
[196/247] Downloading variable 396076 (P3309): Unknown
  Total records for variable 396076: 379; level: 5
  page 1 -> got 100 records; total so far: 200
  page 2 -> got 100 records; total so far: 300
  page 3 -> got 79 records; total so far: 379
  ✓ Successfully collected 379 records (expected: 379)
  ✓ Variable 396076 downloaded (379 records). Total records: 273285
[197/247] Downloading variable 396077 (P3309): Unknown
  Total records for variable 396077: 379; level: 5
  page 1 -> got 100 records; total so far: 200
  page 2 -> got 100 records; total so far: 300
  page 3 -> got 79 records; total so far: 379
  ✓ Successfully collected 379 records (expected: 379)
  ✓ Variable 396077 downloaded (379 records). Total records: 273664
[198/247] Downloading variable 396078 (P3309): Unknown
  Total records for variable 396078: 379; level: 5
Variable 396078, page 3: Error with API key 0: Cannot connect to host bdl.stat.gov.pl:443 ssl:default [Connect call failed ('194.165.48.118', 443)]
  page 1 -> got 100 records; total so far: 200
  page 2 -> got 100 records; total so far: 300
  page 3 -> got 79 records; total so far: 379
  ✓ Successfully collected 379 records (expected: 379)
  ✓ Variable 396078 downloaded (379 records). Total records: 274043
[199/247] Downloading variable 396079 (P3309): Unknown
  Variable 396079: Failed to get initial data: Cannot connect to host bdl.stat.gov.pl:443 ssl:default [Connect call failed ('194.165.48.118', 443)]
  FAILED - var_id: 396079, page: 0, page_size: 100
  WARNING: No data returned for variable 396079
[200/247] Downloading variable 396080 (P3309): Unknown
  Variable 396080: Failed to get initial data: Cannot connect to host bdl.stat.gov.pl:443 ssl:default [Connect call failed ('194.165.48.118', 443)]
  FAILED - var_id: 396080, page: 0, page_size: 100
  WARNING: No data returned for variable 396080
[201/247] Downloading variable 396081 (P3309): Unknown
'''

In [23]:
df_census_data

,id,name,values,variableId,subjectId
0,011212001011,Bochnia,"[{'year': '2021', 'val': 27867, 'attrId': 1}]",1644511,P4253
1,011212001022,Bochnia,"[{'year': '2021', 'val': 19807, 'attrId': 1}]",1644511,P4253
2,011212001032,Drwinia,"[{'year': '2021', 'val': 6214, 'attrId': 1}]",1644511,P4253
3,011212001042,Lipnica Murowana,"[{'year': '2021', 'val': 5358, 'attrId': 1}]",1644511,P4253
4,011212001052,Łapanów,"[{'year': '2021', 'val': 8226, 'attrId': 1}]",1644511,P4253
...,...,...,...,...,...
301577,071427338032,Puszcza Mariańska,"[{'year': '2021', 'val': 3.35, 'attrId': 1}]",1652556,P4287
301578,071427338042,Radziejowice,"[{'year': '2021', 'val': 3.28, 'attrId': 1}]",1652556,P4287
301579,071427338053,Wiskitki,"[{'year': '2021', 'val': 3.54, 'attrId': 1}]",1652556,P4287
301580,071427338054,Wiskitki - miasto,"[{'year': '2021', 'val': 3.42, 'attrId': 1}]",1652556,P4287
